# Spearman Correlation Heatmaps (Within-Group)

Within-group Spearman correlation analysis to identify redundant metrics per category.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns
from pathlib import Path
from scipy.stats import spearmanr

In [ ]:
# ── Categories & colors (same as results_table.ipynb) ──────────────────────
CAT_COLORS = {
    'Complexity':        '#FFF2CC',
    'Cognitive Effort':  '#DDEBF7',
    'State Management':  '#E2EFDA',
    'Size & Verbosity':  '#FCE4D6',
    'Quality':           '#EAD1DC',
}

METRIC_CATEGORIES = {
    'Complexity': [
        'sq_complexity', 'sq_cognitive_complexity',
        'hal_volume', 'hal_difficulty',
    ],
    'Cognitive Effort': [
        'hal_effort', 'hal_time_seconds', 'hal_bugs_delivered',
    ],
    'State Management': [
        'state_mutable_vars', 'state_immutable_vars',
        'state_observable_state_vars', 'state_state_updates',
        'derived_mutable_variable_ratio',
    ],
    'Size & Verbosity': [
        'sq_ncloc', 'sq_lines', 'sq_files', 'sq_functions',
        'hal_length', 'hal_loc',
    ],
    'Quality': [
        'sq_code_smells', 'sq_bugs', 'sq_sqale_debt_ratio',
        'sq_duplicated_lines_density', 'derived_maintainability_index',
    ],
}

In [ ]:
# ── Data loading ───────────────────────────────────────────────────────────
df = pd.read_csv('../results/metrics_table.csv').set_index(['framework', 'project'])
print(f'Loaded {len(df)} projects, {len(df.columns)} metrics')
df.head()

In [ ]:
# ── Within-group Spearman heatmaps ─────────────────────────────────────────

def lighten(hex_color, amount=0.35):
    """Lighten a hex color toward white by `amount` (0=no change, 1=white)."""
    rgb = np.array(mcolors.to_rgb(hex_color))
    return mcolors.to_hex(rgb + (1 - rgb) * amount)


def spearman_heatmap(ax, subset, title, bg_color):
    """Compute and draw a Spearman correlation heatmap on `ax`."""
    corr = subset.corr(method='spearman')

    # p-value mask (mark |r| < threshold or p > 0.05 with hatching)
    n = len(subset)
    mask_insignificant = pd.DataFrame(False, index=corr.index, columns=corr.columns)
    for c1 in corr.columns:
        for c2 in corr.columns:
            if c1 == c2:
                continue
            valid = subset[[c1, c2]].dropna()
            if len(valid) < 4:
                mask_insignificant.loc[c1, c2] = True
                continue
            _, p = spearmanr(valid[c1], valid[c2])
            if p > 0.05:
                mask_insignificant.loc[c1, c2] = True

    # Build diverging colormap anchored at the category background color
    cmap = sns.diverging_palette(220, 20, as_cmap=True)

    sns.heatmap(
        corr,
        ax=ax,
        cmap=cmap,
        vmin=-1, vmax=1,
        annot=True, fmt='.2f', annot_kws={'size': 8},
        linewidths=0.5, linecolor='white',
        square=True,
        cbar_kws={'shrink': 0.8, 'label': 'Spearman ρ'},
    )

    # Cross-hatch insignificant cells
    for i, row_name in enumerate(corr.index):
        for j, col_name in enumerate(corr.columns):
            if mask_insignificant.loc[row_name, col_name]:
                ax.add_patch(plt.Rectangle(
                    (j, i), 1, 1,
                    fill=False, hatch='////', edgecolor='grey',
                    linewidth=0, alpha=0.5
                ))

    ax.set_title(title, fontsize=12, fontweight='bold', pad=10)
    ax.set_facecolor(lighten(bg_color))
    ax.tick_params(axis='x', rotation=30, labelsize=8)
    ax.tick_params(axis='y', rotation=0,  labelsize=8)


n_cats  = len(METRIC_CATEGORIES)
fig, axes = plt.subplots(1, n_cats, figsize=(5 * n_cats, 5))
fig.suptitle(
    'Within-Group Spearman Correlation\n'
    '(hatched = p > 0.05)',
    fontsize=14, y=1.02
)

for ax, (cat, metrics) in zip(axes, METRIC_CATEGORIES.items()):
    available = [m for m in metrics if m in df.columns]
    subset = df[available].dropna(how='all')
    spearman_heatmap(ax, subset, cat, CAT_COLORS[cat])

plt.tight_layout()
#plt.savefig('../results/spearman_heatmap_within_group.png',
          #  dpi=150, bbox_inches='tight')
plt.show()
print('Saved → results/spearman_heatmap_within_group.png')

In [ ]:
# ── Cross-group Spearman heatmap ───────────────────────────────────────────
# All metrics ordered by category; axis labels colored by category.

# Build ordered metric list and matching color list
ordered_metrics = []
tick_colors     = []
for cat, metrics in METRIC_CATEGORIES.items():
    available = [m for m in metrics if m in df.columns]
    ordered_metrics.extend(available)
    tick_colors.extend([CAT_COLORS[cat]] * len(available))

subset = df[ordered_metrics]
corr   = subset.corr(method='spearman')

# Significance mask
from scipy.stats import spearmanr
from itertools import combinations

sig_mask = pd.DataFrame(False, index=corr.index, columns=corr.columns)
for m1, m2 in combinations(ordered_metrics, 2):
    valid = subset[[m1, m2]].dropna()
    if len(valid) < 4:
        sig_mask.loc[m1, m2] = True
        sig_mask.loc[m2, m1] = True
        continue
    _, p = spearmanr(valid[m1], valid[m2])
    if p > 0.05:
        sig_mask.loc[m1, m2] = True
        sig_mask.loc[m2, m1] = True

n = len(ordered_metrics)
fig, ax = plt.subplots(figsize=(n * 0.55 + 2, n * 0.55 + 2))

cmap = sns.diverging_palette(220, 20, as_cmap=True)
sns.heatmap(
    corr,
    ax=ax,
    cmap=cmap,
    vmin=-1, vmax=1,
    annot=True, fmt='.2f', annot_kws={'size': 6},
    linewidths=0.3, linecolor='white',
    square=True,
    cbar_kws={'shrink': 0.6, 'label': 'Spearman ρ'},
)

# Cross-hatch insignificant cells
for i, r in enumerate(corr.index):
    for j, c in enumerate(corr.columns):
        if sig_mask.loc[r, c]:
            ax.add_patch(plt.Rectangle(
                (j, i), 1, 1,
                fill=False, hatch='////', edgecolor='grey',
                linewidth=0, alpha=0.4
            ))

# Color tick labels by category
for tick, color in zip(ax.get_xticklabels(), tick_colors):
    tick.set_color('black')
    tick.set_bbox(dict(boxstyle='round,pad=0.15', facecolor=color,
                       edgecolor='none', alpha=0.8))
for tick, color in zip(ax.get_yticklabels(), tick_colors):
    tick.set_color('black')
    tick.set_bbox(dict(boxstyle='round,pad=0.15', facecolor=color,
                       edgecolor='none', alpha=0.8))

ax.tick_params(axis='x', rotation=40, labelsize=7)
ax.tick_params(axis='y', rotation=0,  labelsize=7)

# Draw category group separators
pos = 0
for cat, metrics in METRIC_CATEGORIES.items():
    size = len([m for m in metrics if m in df.columns])
    if pos > 0:
        ax.axhline(pos, color='black', linewidth=1.5)
        ax.axvline(pos, color='black', linewidth=1.5)
    pos += size

# Legend patches
import matplotlib.patches as mpatches
legend_patches = [
    mpatches.Patch(facecolor=color, edgecolor='grey', label=cat)
    for cat, color in CAT_COLORS.items()
]
ax.legend(handles=legend_patches, bbox_to_anchor=(1.18, 1), loc='upper left',
          fontsize=8, title='Category', title_fontsize=8, framealpha=0.9)

ax.set_title('Cross-Group Spearman Correlation (hatched = p > 0.05)',
             fontsize=12, fontweight='bold', pad=12)

plt.tight_layout()
plt.savefig('../results/spearman_heatmap_cross_group.png',
            dpi=150, bbox_inches='tight')
plt.show()
print('Saved → results/spearman_heatmap_cross_group.png')


In [ ]:
# ── Group-to-group summary heatmap (mean |ρ| between groups) ──────────────
cats = list(METRIC_CATEGORIES.keys())
group_corr = pd.DataFrame(index=cats, columns=cats, dtype=float)

for cat_a in cats:
    for cat_b in cats:
        ma = [m for m in METRIC_CATEGORIES[cat_a] if m in df.columns]
        mb = [m for m in METRIC_CATEGORIES[cat_b] if m in df.columns]
        rhos = []
        for m1 in ma:
            for m2 in mb:
                if m1 == m2:
                    continue
                valid = df[[m1, m2]].dropna()
                if len(valid) < 4:
                    continue
                rho, p = spearmanr(valid[m1], valid[m2])
                if p <= 0.05:
                    rhos.append(abs(rho))
        group_corr.loc[cat_a, cat_b] = round(np.mean(rhos), 3) if rhos else np.nan

fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(
    group_corr.astype(float),
    ax=ax,
    cmap='YlOrRd',
    vmin=0, vmax=1,
    annot=True, fmt='.2f', annot_kws={'size': 10},
    linewidths=1, linecolor='white',
    square=True,
    cbar_kws={'shrink': 0.7, 'label': 'mean |ρ| (significant pairs only)'},
)

# Color axis tick labels
for tick in ax.get_xticklabels():
    tick.set_bbox(dict(boxstyle='round,pad=0.2',
                       facecolor=CAT_COLORS[tick.get_text()],
                       edgecolor='none', alpha=0.9))
for tick in ax.get_yticklabels():
    tick.set_bbox(dict(boxstyle='round,pad=0.2',
                       facecolor=CAT_COLORS[tick.get_text()],
                       edgecolor='none', alpha=0.9))

ax.tick_params(axis='x', rotation=30, labelsize=9)
ax.tick_params(axis='y', rotation=0,  labelsize=9)
ax.set_title('Group-to-Group Agreement\n(mean |ρ| of significant pairs)',
             fontsize=12, fontweight='bold', pad=10)

plt.tight_layout()
plt.savefig('../results/spearman_heatmap_group_summary.png',
            dpi=150, bbox_inches='tight')
plt.show()
print('Saved → results/spearman_heatmap_group_summary.png')


In [ ]:
# ── Tabular summary: pairs with |ρ| ≥ 0.8 (potentially redundant) ──────────
from itertools import combinations

REDUNDANCY_THRESHOLD = 0.8
records = []

for cat, metrics in METRIC_CATEGORIES.items():
    available = [m for m in metrics if m in df.columns]
    subset = df[available].dropna(how='all')
    corr = subset.corr(method='spearman')

    for m1, m2 in combinations(available, 2):
        valid = subset[[m1, m2]].dropna()
        if len(valid) < 4:
            continue
        rho, p = spearmanr(valid[m1], valid[m2])
        if abs(rho) >= REDUNDANCY_THRESHOLD:
            records.append({
                'Category': cat,
                'Metric A': m1,
                'Metric B': m2,
                'ρ': round(rho, 3),
                'p-value': round(p, 4),
                'n': len(valid),
            })

redundant_df = pd.DataFrame(records).sort_values(['Category', 'ρ'], ascending=[True, False])
print(f'Pairs with |ρ| ≥ {REDUNDANCY_THRESHOLD}: {len(redundant_df)}')
redundant_df